# Upload IRIS FDM data to filament

Uploads a level of a mosaic to filament so that `iris_prep` can be applied there.
Used twice per mosaic: level 1 (before `apply_iris_prep_through_bg_subtraction.pro`)
and level 1.2 (before `apply_remaining_iris_prep.pro`).

Authentication is by SSH key via the `filament` host alias in `~/.ssh/config`.
**Never put a password in this notebook.**

> Storage note: `/disk/data` on filament runs at capacity. Delete the previous
> level from filament once the next level has been produced and downloaded.


In [ ]:
import pathlib as pl
import subprocess


Set the mosaic date and which level is being uploaded.


In [ ]:
date_string = '20240811'
level = 'level_12'   # 'level_1' before iris_prep part 1; 'level_12' before part 2

local_path = pl.Path(fr'D:\IRIS data\deep_mosaics\{date_string}\{level}')
remote_host = 'filament'   # host alias from ~/.ssh/config (key auth, no password)
remote_path = f'/disk/data/cbunn/calibrated_iris_mosaics/deep_mosaics/{date_string}/{level}'

local_files = sorted(local_path.glob('*.fits'))
print(f'{len(local_files)} files, {sum(f.stat().st_size for f in local_files) / 1e9:.1f} GB')


Check that the destination has room before starting. `/disk/data` is shared and frequently near capacity.


In [ ]:
free = subprocess.run(
    ['ssh', remote_host, 'sh', '-c', "'df -B1 --output=avail /disk/data | tail -1'"],
    capture_output=True, text=True, check=True,
).stdout.strip()
needed = sum(f.stat().st_size for f in local_files)
print(f'need {needed/1e9:.1f} GB, free {int(free)/1e9:.1f} GB')
assert int(free) > needed, 'not enough free space on filament'


Create the destination directory and upload.

`scp` is used here because Windows OpenSSH ships it by default. If `rsync` is
installed locally, prefer it — it resumes after an interrupted transfer, which
matters because the campus VPN can drop during a multi-hour upload:

```
rsync -avP --partial <local>/ filament:<remote>/
```


In [ ]:
subprocess.run(
    ['ssh', remote_host, 'sh', '-c', f"'mkdir -p {remote_path}'"],
    check=True,
)


In [ ]:
%%time
subprocess.run(
    ['scp', *[str(f) for f in local_files], f'{remote_host}:{remote_path}/'],
    check=True,
)


Verify the file count matches before moving on.


In [ ]:
remote_count = subprocess.run(
    ['ssh', remote_host, 'sh', '-c', f"'ls -1 {remote_path}/*.fits | wc -l'"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print(f'local {len(local_files)}, remote {remote_count}')
assert int(remote_count) == len(local_files), 'file count mismatch'
